# Micro-pilot: C4 Ablation Sweep

**Goal:** identify meta-heads — attention heads whose ablation disrupts self-specific prediction (Δ_self) more than task performance (Δ_task) or cross-model prediction (Δ_cross).

**Design:**
- Pythia 410M step 143000 (same as Paper 2 final checkpoint)
- 144 head ablations (heads $\{0, 3, 6, 9, 12, 15\}$ × 24 layers) — identity-tracked, same as Paper 2
- Pre-computed Pythia 1.4B Phase B answers (frozen) — cross-alignment baseline
- 40-question bank filtered for robust disagreement (produced by `build_question_bank.ipynb`)

**Three measurements per ablation:**
1. **Δ_self**: change in `Pythia_A → Pythia_B` match rate on disagreement subset (self-specific signal)
2. **Δ_cross**: change in `Pythia_A → Pythia_1.4B_B` match rate (general Q-answering control)
3. **Δ_task**: change in validation loss on wikitext-103 (Paper 2's task metric)

**Meta-head signature:** high Δ_self, low Δ_cross, low Δ_task.

**Gating criteria for Phase 1 launch:**
- At least 5 heads with Δ_self significantly > Δ_task
- At least 7 out of top-10 heads by Δ_self show Δ_self > Δ_cross

**Compute:** ~18-22h on A100 40GB, $20-25.

**Resume:** saves per-head after each ablation. If Colab disconnects, re-run — pre-computed rows are skipped.

**Prerequisite:** upload `question_bank.json` (from previous notebook) to Drive's `DFE_research/preflight/` or local `./preflight/`.

In [ ]:
!pip install -q transformers datasets torch accelerate pandas

In [ ]:
import torch, json, os, time, csv, hashlib, gc
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name()}  |  Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# MANDATORY Drive mount. Runtime resets will destroy local files; Drive persists.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = '/content/drive/MyDrive/DFE_research/preflight'
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f'Drive mounted; output to {OUT_DIR}')
    # Verify write access
    test_file = os.path.join(OUT_DIR, '.write_test')
    with open(test_file, 'w') as f: f.write('ok')
    os.remove(test_file)
    print('Drive write verified.')
except Exception as e:
    raise RuntimeError(
        f'Drive mount or write failed: {e}\n\n'
        'This notebook REQUIRES Drive to persist results across potential runtime disconnects.\n'
        'Steps: (1) authorize Drive access in the popup, (2) if popup blocked, '
        'click the folder icon in left sidebar -> mount Drive, (3) re-run this cell.'
    )

ABLATION_CSV = os.path.join(OUT_DIR, 'micropilot_ablations.csv')
CSV_FIELDS = ['layer_idx', 'head_idx', 'baseline_self', 'ablated_self',
              'baseline_cross', 'ablated_cross', 'baseline_task', 'ablated_task',
              'delta_self', 'delta_cross', 'delta_task', 'elapsed_sec']

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

def append_row(row):
    new = not os.path.exists(ABLATION_CSV)
    with open(ABLATION_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if new: w.writeheader()
        w.writerow(row)

## Load question bank

In [ ]:
import urllib.request

GITHUB_RAW = 'https://raw.githubusercontent.com/mool32/functional-differentiation-dfe/main'

qb_path = os.path.join(OUT_DIR, 'question_bank.json')
if not os.path.exists(qb_path):
    log('Downloading question bank from GitHub...')
    url = f'{GITHUB_RAW}/data/micropilot/question_bank.json'
    urllib.request.urlretrieve(url, qb_path)
    log(f'  Downloaded to {qb_path}')

with open(qb_path) as f:
    qb = json.load(f)

QUESTIONS = qb['questions']
N_QUESTIONS = len(QUESTIONS)
log(f'Loaded question bank: {N_QUESTIONS} questions')

from collections import Counter
dc = Counter(q['domain'] for q in QUESTIONS)
print('Domain distribution:', dict(dc))

# Show a few examples
print('\nFirst 3 questions:')
for q in QUESTIONS[:3]:
    print(f'  [{q["domain"]}] {q["q"][:50]}')
    strength = q.get('strength_410', q.get('strength', 0))
    print(f'    410M->{q["answer_410m"]}, 1.4B->{q["answer_14b"]}, strength={strength:.1f}')

## Load model and tokenizer

In [ ]:
log('Loading Pythia 410M step 143000 (float32 + TF32)...')
MODEL_NAME = 'EleutherAI/pythia-410m-deduped'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision='step143000', torch_dtype=torch.float32
).to(device).eval()

# Architecture constants
N_LAYERS = 24
FIXED_HEADS = [0, 3, 6, 9, 12, 15]  # 144 heads total, matching Paper 2
HEAD_DIM = model.config.hidden_size // model.config.num_attention_heads
log(f'Layers: {N_LAYERS}, heads per layer: {model.config.num_attention_heads}, head dim: {HEAD_DIM}')

## Prompts + classification utilities

In [ ]:
PHASE_A = '''If I asked you 'What is 5 + 3?', your first word would be: Eight
If I asked you 'Who wrote Hamlet?', your first word would be: Shakespeare
If I asked you 'What color is grass?', your first word would be: Green
If I asked you '{q}', your first word would be:'''

@torch.no_grad()
def get_top_token_str(prompt):
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    logits = model(**inputs).logits[0, -1, :]
    return tok.decode([logits.argmax().item()]).strip().lower()

def compute_alignment_scores(questions):
    """Run Phase A on current model state, return:
    - self_rate: fraction where Pythia_A matches Pythia_B (from question bank's answer_410m)
    - cross_rate: fraction where Pythia_A matches 1.4B_B (from answer_14b)
    Both computed on the full question set (all are disagreement cases)."""
    n_self, n_cross = 0, 0
    for q in questions:
        pred = get_top_token_str(PHASE_A.format(q=q['q']))
        if pred == q['answer_410m']:
            n_self += 1
        if pred == q['answer_14b']:
            n_cross += 1
    return n_self / len(questions), n_cross / len(questions)

## Task loss evaluation (wikitext-103, matching Paper 2)

In [ ]:
from datasets import load_dataset

EVAL_N_BATCHES = 25
EVAL_BATCH_SIZE = 4
EVAL_SEQ_LEN = 2048

tokens_path = os.path.join(OUT_DIR, 'validation_tokens_micropilot.pt')
if os.path.exists(tokens_path):
    log('Loading cached validation tokens...')
    token_batches = torch.load(tokens_path, weights_only=True)
else:
    log('Streaming wikitext-103 train for validation...')
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)
    all_toks = []
    need = EVAL_N_BATCHES * EVAL_BATCH_SIZE * EVAL_SEQ_LEN
    for ex in ds:
        t = ex.get('text', '').strip()
        if len(t) < 50: continue
        ids = tok(t, return_tensors='pt', truncation=False)['input_ids'].squeeze()
        if ids.dim() == 0: continue
        all_toks.append(ids)
        if sum(x.numel() for x in all_toks) >= need * 1.2: break
    all_toks = torch.cat(all_toks)[:need]
    token_batches = all_toks.reshape(EVAL_N_BATCHES, EVAL_BATCH_SIZE, EVAL_SEQ_LEN)
    torch.save(token_batches, tokens_path)
log(f'Validation tokens: {token_batches.shape}')

@torch.no_grad()
def evaluate_task_loss():
    total = 0.0
    for i in range(token_batches.shape[0]):
        ids = token_batches[i].to(device)
        total += model(input_ids=ids, labels=ids).loss.item()
    return total / token_batches.shape[0]

## Ablation + restore (same as Paper 2 protocol)

In [ ]:
def ablate_head(layer_idx, head_idx):
    s, e = head_idx * HEAD_DIM, (head_idx + 1) * HEAD_DIM
    w = model.gpt_neox.layers[layer_idx].attention.dense.weight
    saved = w.data.clone()
    w.data[:, s:e] = 0
    return saved

def restore_head(layer_idx, saved):
    model.gpt_neox.layers[layer_idx].attention.dense.weight.data.copy_(saved)

# Sanity check: bitwise save/restore
import hashlib
def tensor_hash(t):
    return hashlib.sha256(t.detach().cpu().contiguous().numpy().tobytes()).hexdigest()[:16]

w_ref = model.gpt_neox.layers[5].attention.dense.weight
h0 = tensor_hash(w_ref.data)
s = ablate_head(5, 7)
h1 = tensor_hash(w_ref.data)
restore_head(5, s)
h2 = tensor_hash(w_ref.data)
assert h0 == h2 and h0 != h1, 'save/restore broken!'
log(f'Save/restore verified: before={h0}, ablated={h1}, restored={h2}')

## Baseline measurements (no ablation)

Compute baseline values once. All Δ values are measured against these.

In [ ]:
baseline_path = os.path.join(OUT_DIR, 'micropilot_baseline.json')
if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        baseline = json.load(f)
    log(f'Loaded cached baseline: {baseline}')
else:
    log('Computing baseline (self_rate, cross_rate, task_loss) without ablation...')
    t0 = time.time()
    baseline_self, baseline_cross = compute_alignment_scores(QUESTIONS)
    baseline_task = evaluate_task_loss()
    baseline = {
        'self_rate': baseline_self,
        'cross_rate': baseline_cross,
        'task_loss': baseline_task,
        'elapsed_sec': time.time() - t0,
    }
    with open(baseline_path, 'w') as f:
        json.dump(baseline, f, indent=2)
    log(f'Baseline: self={baseline_self:.3f}  cross={baseline_cross:.3f}  task={baseline_task:.4f}')
    log(f'  (elapsed: {baseline["elapsed_sec"]:.1f}s)')

## Main ablation sweep

For each of 144 heads: ablate → measure self, cross, task → restore → record.

Resume: skips heads already in CSV.

In [ ]:
# Load existing progress
completed = set()
if os.path.exists(ABLATION_CSV):
    import pandas as pd
    df_existing = pd.read_csv(ABLATION_CSV)
    completed = set((int(r['layer_idx']), int(r['head_idx'])) for _, r in df_existing.iterrows())
    log(f'CSV has {len(completed)} completed ablations; will skip.')

todo = [(L, H) for L in range(N_LAYERS) for H in FIXED_HEADS if (L, H) not in completed]
log(f'To do: {len(todo)} ablations')

total_sweep_start = time.time()
for idx, (L, H) in enumerate(todo):
    t0 = time.time()
    saved = ablate_head(L, H)
    ablated_self, ablated_cross = compute_alignment_scores(QUESTIONS)
    ablated_task = evaluate_task_loss()
    restore_head(L, saved)
    elapsed = time.time() - t0

    row = {
        'layer_idx': L, 'head_idx': H,
        'baseline_self': baseline['self_rate'], 'ablated_self': ablated_self,
        'baseline_cross': baseline['cross_rate'], 'ablated_cross': ablated_cross,
        'baseline_task': baseline['task_loss'], 'ablated_task': ablated_task,
        'delta_self': baseline['self_rate'] - ablated_self,   # positive = ablation hurt self-alignment
        'delta_cross': baseline['cross_rate'] - ablated_cross,
        'delta_task': ablated_task - baseline['task_loss'],   # positive = loss increased
        'elapsed_sec': elapsed,
    }
    append_row(row)

    # Verbose every head for first 5, then every 10
    if idx < 5 or (idx + 1) % 10 == 0:
        log(f'  [{idx+1}/{len(todo)}] L{L:>2}H{H:>2}: Δself={row["delta_self"]:+.3f}  Δcross={row["delta_cross"]:+.3f}  Δtask={row["delta_task"]:+.4f}  ({elapsed:.0f}s)')

log(f'\nSweep complete: {len(todo)} ablations in {(time.time()-total_sweep_start)/3600:.1f} hours')

## Analysis — meta-head candidates

In [ ]:
import pandas as pd

df = pd.read_csv(ABLATION_CSV)
log(f'Loaded {len(df)} ablations for analysis')

# Rank by Δ_self
df_sorted = df.sort_values('delta_self', ascending=False).reset_index(drop=True)

print('\n=== TOP 15 HEADS BY Δ_self ===')
print(f"{'L':>3} {'H':>3} | {'Δself':>7} {'Δcross':>7} {'Δtask':>7} | Meta signature?")
print('-' * 65)
for i, r in df_sorted.head(15).iterrows():
    meta = '✓' if (r['delta_self'] > r['delta_cross'] and r['delta_self'] > 5 * r['delta_task']) else ' '
    print(f'{int(r["layer_idx"]):>3} {int(r["head_idx"]):>3} | {r["delta_self"]:>+7.3f} {r["delta_cross"]:>+7.3f} {r["delta_task"]:>+7.4f} | {meta}')

# Gating criteria
# Criterion 1: at least 5 heads with Δ_self > 2 × Δ_task (both scaled somehow)
# Δ_task is in loss units (0.001 scale), Δ_self is in rate units (0-1). Use ratio Δ_self / Δ_task.
# Rough equivalence: Δ_task of 0.01 = 0.3% relative loss change. Δ_self of 0.05 = 5pp drop. 5/0.3 ~ 17x scale diff.
# Use simpler criterion: Δ_self > 0.05 (5pp drop) AND Δ_task < 0.02 (small task impact)
meta_heads = df[(df['delta_self'] > 0.05) & (df['delta_task'] < 0.02)]
crit1 = len(meta_heads) >= 5

# Criterion 2: of top-10 by Δ_self, at least 7 must have Δ_self > Δ_cross
top10 = df_sorted.head(10)
n_self_gt_cross = sum(1 for _, r in top10.iterrows() if r['delta_self'] > r['delta_cross'])
crit2 = n_self_gt_cross >= 7

print('\n=== GATING CRITERIA ===')
print(f'Criterion 1 (≥5 heads with Δ_self>0.05 AND Δ_task<0.02): {len(meta_heads)} heads  → {"PASS" if crit1 else "FAIL"}')
print(f'Criterion 2 (≥7 of top-10 by Δ_self have Δ_self>Δ_cross): {n_self_gt_cross}/10  → {"PASS" if crit2 else "FAIL"}')
print()
if crit1 and crit2:
    print('>>> PHASE 1 GO: meta-heads detected with clean signature. <<<')
else:
    print('>>> PHASE 1 NO-GO: signal is diffuse or absent at head granularity. <<<')
    print('    Options: (a) N-head ablation, (b) instruction-tuned model, (c) short negative-result note.')

## Paper 2 classification cross-reference

Overlay meta-heads on Paper 2's four-way classification (born-critical / emergent / growing / never-critical).

In [ ]:
# Auto-download Paper 2 ablation data for cross-reference
paper2_local = os.path.join(OUT_DIR, 'paper2_all_ablations.csv')
if not os.path.exists(paper2_local):
    log('Downloading Paper 2 ablation data from GitHub...')
    url = f'{GITHUB_RAW}/data/all_ablations.csv'
    try:
        urllib.request.urlretrieve(url, paper2_local)
        log(f'  Downloaded to {paper2_local}')
    except Exception as e:
        log(f'  Download failed: {e}')
        paper2_local = None

paper2_df = None
if paper2_local and os.path.exists(paper2_local):
    paper2_df = pd.read_csv(paper2_local)
    log(f'Loaded Paper 2 ablations: {len(paper2_df)} rows')

if paper2_df is None:
    log('Paper 2 data not available — skipping cross-reference.')
else:
    # Build classification map from Paper 2 head ablations
    p2 = paper2_df[paper2_df['perturbation_type'] == 'head'].copy()
    classes = {}
    for (L, H), g in p2.groupby(['layer_idx', 'head_idx']):
        g = g.sort_values('checkpoint')
        init = abs(g.iloc[0]['delta'])
        final = abs(g.iloc[-1]['delta'])
        if final < 5e-4:
            cls = 'never-critical'
        elif init > 5e-4 and final > 5e-4:
            cls = 'born-critical'
        elif init < 1e-4 and final > 5e-4:
            cls = 'emergent'
        else:
            cls = 'growing'
        classes[(int(L), int(H))] = cls

    # Join meta-head ranking with Paper 2 class
    df_sorted['paper2_class'] = df_sorted.apply(
        lambda r: classes.get((int(r['layer_idx']), int(r['head_idx'])), 'unknown'), axis=1
    )

    print('=== TOP 15 META-HEAD CANDIDATES x PAPER 2 CLASS ===')
    print(f"{'L':>3} {'H':>3} | {'Dself':>7} {'Dcross':>7} | {'Paper 2 class':<15}")
    print('-' * 55)
    for i, r in df_sorted.head(15).iterrows():
        print(f'{int(r["layer_idx"]):>3} {int(r["head_idx"]):>3} | {r["delta_self"]:>+7.3f} {r["delta_cross"]:>+7.3f} | {r["paper2_class"]:<15}')

    # Distribution of top-15 meta-heads across Paper 2 classes
    print('\nTop-15 meta-heads distribution across Paper 2 classes:')
    print(df_sorted.head(15)['paper2_class'].value_counts().to_string())

    # Enrichment analysis
    top15_classes = df_sorted.head(15)['paper2_class'].value_counts()
    paper2_class_sizes = pd.Series(classes.values()).value_counts()
    print('\nEnrichment analysis (top-15 observed vs expected given class size):')
    for cls, observed in top15_classes.items():
        expected = 15 * paper2_class_sizes.get(cls, 0) / sum(paper2_class_sizes)
        ratio = observed / expected if expected > 0 else float('inf')
        print(f'  {cls:<15}: observed {observed}, expected {expected:.1f}, ratio {ratio:.2f}x')

## Save final results

In [ ]:
summary = {
    'baseline': baseline,
    'n_ablations': len(df),
    'gating_criteria': {
        'crit1_meta_count': len(meta_heads),
        'crit1_pass': bool(crit1),
        'crit2_top10_self_gt_cross': int(n_self_gt_cross),
        'crit2_pass': bool(crit2),
        'overall_pass': bool(crit1 and crit2),
    },
    'top15_meta_candidates': [
        {
            'layer': int(r['layer_idx']),
            'head': int(r['head_idx']),
            'delta_self': float(r['delta_self']),
            'delta_cross': float(r['delta_cross']),
            'delta_task': float(r['delta_task']),
            'paper2_class': r.get('paper2_class', 'unknown'),
        }
        for _, r in df_sorted.head(15).iterrows()
    ],
}
summary_path = os.path.join(OUT_DIR, 'micropilot_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

log(f'Saved summary: {summary_path}')
log(f'Raw ablations: {ABLATION_CSV}')
log(f'Download both for analysis/write-up.')

## Force-download results (insurance against runtime reset)

Files are already on Drive, but also download to browser now for double safety.

In [ ]:
# Also download to browser immediately (belt and suspenders)
try:
    from google.colab import files
    print('Triggering browser downloads...')
    files.download(ABLATION_CSV)
    files.download(summary_path)
    baseline_path_local = os.path.join(OUT_DIR, 'micropilot_baseline.json')
    if os.path.exists(baseline_path_local):
        files.download(baseline_path_local)
    print('Done. Check your browser download bar.')
except Exception as e:
    print(f'Download failed: {e}')
    print(f'Files still on Drive at {OUT_DIR}/')
    print(f'Manual retrieval: open Drive, navigate to DFE_research/preflight/')